## 20_Read_CT_Data
<b>Env</b>: cdms_plus (Python 3.12.7)

<br>
<hr style="border:1px solid blue">


#### Imports

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import time
from pathlib import Path

pd.options.display.max_columns=30
start_time = time.time()

#### Initialisation and settings

In [ ]:
drive_letter = 'C:/'
proj_root = drive_letter + '@Adnan/'

file_input = proj_root + 'Data_Out_Peak_Amp/'                       # Cyclic Top peak amplitude files

engine = 'pyarrow'                  # Parquet engine to use
comp = 'snappy'                     # Compression format

METRES_TO_YARDS = 1.0936133

ANH_DEBUG = False                   # Debug - ie, output CSVs etc

CT_THRESHOLD = 5                    # Peak Amplitude Threshold (mm)
UPMULTIPLIER = 0.8                  # If the Peak Amplitude is above the CT_THRESHOLD, then multiply by this before Accumulating 

In [ ]:
def parquet_filter_by_elr(data,elr,cols=None):

    # data: root location of existing file
    # filename: filter data for this filename

    filter = [('ELR', '=', elr)]

    dataset = pq.ParquetDataset(data, filters=filter)   # Details of reading the dataset from multiple files and partitions

    if cols is None:
        table = dataset.read()              # Reads all columns
    else:
        table = dataset.read(columns=cols)  # Read using the column list

    df = table.to_pandas()                  # Convert to a pandas dataframe

    return df

# Example use: 'df = parquet_filter_by_elr(data=file_input, elr='CRR1')'

In [ ]:
def get_elrs():

    '''
    Get the filenames from the partioned parquet file
    The file is partitioned 'Ref', 'ELR', then 'TrackID

    Return a list of filenames
    '''

    filenames = [
        p.name.split("=", 1)[1]
        for p in Path(file_input).iterdir()
        if p.is_dir() and p.name.startswith("ELR=")
    ]

    return filenames

In [ ]:
def CT_category(amplitude8):

    # read the peak amplitude, multiplied by 0.8, and return the cyclic top catergorisation
    # If it is pre-alert assing a value of 'Pre'

    #----------------------------------------------------------------------
    # Cat	Severity	Threshold   Threshold	Rails	 
    #                   From	    To
    #----------------------------------------------------------------------
    # Pre                           18 
    # D	    AL	        18	        20	        One	
    # C	    IL	        20	        23	        One	
    # B	    IL	        23	        26	        One	    
    # A	    IL	        26	        30	        One	
    # I	    IAL	        30	        1000	    One	(1000 just for lookup)
    #----------------------------------------------------------------------

    if amplitude8 < 18:
        return 'Pre'
    elif amplitude8 < 20:
        return 'D'
    elif amplitude8 < 23:
        return 'C'
    elif amplitude8 < 26:
        return 'B'
    elif amplitude8 < 30:
        return 'A'
    elif amplitude8 < 1000:
        return 'I'



In [ ]:
def parquet_filter_by_ref(data,ref,cols=None):

    # data: root location of existing file
    # filename: filter data for this filename

    filter = [('Ref', '=', ref)]

    dataset = pq.ParquetDataset(data, filters=filter)   # Details of reading the dataset from multiple files and partitions

    if cols is None:
        table = dataset.read()              # Reads all columns
    else:
        table = dataset.read(columns=cols)  # Read using the column list

    df = table.to_pandas()                  # Convert to a pandas dataframe

    return df

# Example use: 'df = parquet_filter_by_ref(data=data_in_sine, filename='TRC_Export_131103_20260905_112801')'

In [ ]:
# df = parquet_filter_by_ref(data=file_input,ref='TRC_Export_131103_20260905_112801')
# df

In [ ]:
df = parquet_filter_by_elr(data=file_input, elr='CRR1')
df

In [ ]:
ct_cols = [
    '45_top_right','45_top_left', 
    '6_top_right', '6_top_left', 
    '9_top_right', '9_top_left',
    '135_top_right', '135_top_left', 
    '18_top_right', '18_top_left'
    ]

cols_to_use = ['CT_Index','ELR','TrackID','Mileage','Location_m','Ref','Date', ct_cols[8]]

df_ct_col = df[cols_to_use]
df_ct_col_without_nans = df_ct_col.dropna(subset=[ct_cols[8]])


In [ ]:
df_ct_col_without_nans

In [ ]:
df_ct_col_without_nans['Date'].unique()